# Lab 2: Text Analytics and Spam Detection with PySpark



## Step 1 — Install PySpark and Java

In [ ]:
# Install Java and PySpark
!apt-get update -y >/dev/null 2>&1
!apt-get install -y openjdk-17-jdk >/dev/null 2>&1
!pip install -q pyspark==3.5.1

# Set Java environment variable
import os
# os.environ["JAVA_HOME"] = [insert]

print("Installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
Installation complete.


In [4]:
# Verify Java installation
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


---
## Step 2 — Start a Spark Session

A `SparkSession` is the unified entry point for all Spark functionality. Setting `.master("local[*]")` tells Spark to run locally using all available CPU cores.

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab2_SMS_Analysis") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark is active.")
print("Spark Version:", spark.version)
print(sc)

Spark is active.
Spark Version: 3.5.1
<SparkContext master=local[*] appName=Lab2_SMS_Analysis>



###  Checkpoint 1
**1. What is a Spark Session?**  
A SparkSession is the main entrypoint in code for the Apache Spark engine. It bundles `SparkContext`, `SQLContext`, and `HiveContext` into a single object. I lets you create DataFrames, execute SQL queries, and manage Spark configuration at once.

**2. Why might Spark be more useful than plain Python for large datasets?**  
Normal Python processes data sequentially on a single machine, which becomes less useful once a dataset exceeds the available RAM. Spark distributes the storage and  computation across a cluster of nodes. It can process datasets too large to fit in memory on one machine.


## Step 3 — Upload the Dataset


In [6]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import os

print("Files in current directory:")
print(os.listdir())

You should see **both** `ham.txt` and `spam.txt` listed above before continuing.


## Step 4 — Load Data into Spark RDDs

An **RDD (Resilient Distributed Dataset)** is Spark's foundational data abstraction. `sc.textFile()` reads each line of a text file into a separate element of the RDD. The data is *distributed* — conceptually split across partitions — and Spark can process those partitions in parallel.

In [8]:
ham_rdd  = sc.textFile("/content/ham.txt")
spam_rdd = sc.textFile("/content/spam.txt")

ham_count  = ham_rdd.count()
spam_count = spam_rdd.count()

print(f"Ham messages loaded:  {ham_count}")
print(f"Spam messages loaded: {spam_count}")

Ham messages loaded:  750
Spam messages loaded: 750


In [9]:
print("Sample Ham Messages:")
print(ham_rdd.take(3))

print("\nSample Spam Messages:")
print(spam_rdd.take(3))

Sample Ham Messages:
['Dear Spark Learner, Thanks so much for attending the Spark Summit 2014!  Check out videos of talks from the summit at ...', 'Hi Mom, Apologies for being late about emailing and forgetting to send you the package.  I hope you and bro have been ...', 'Wow, hey Fred, just heard about the Spark petabyte sort.  I think we need to take time to try it out immediately ...']

Sample Spam Messages:
['Dear sir, I am a Prince in a far kingdom you have not heard of.  I want to send you money via wire transfer so please ...', 'Get Viagra real cheap!  Send money right away to ...', 'Oh my gosh you can be really strong too with these drugs found in the rainforest. Get them cheap right now ...']


###  Checkpoint 2 — RDDs vs. Python Lists

**1. What is an RDD?**  
An RDD (Resilient Distributed Dataset) is Spark's core abtracting methofs for representing a collection of data elements that are partitioned across the nodes of a cluste. The "resilient" part means Spark can automatically reconstruct lost partitions by replaying the original transformation structure.

**2. How is an RDD different from a normal Python list?**  
A Python list lives  in the memory of a single process on a single machine. If  the data is too large to fit in RAM, the program  will crash.
On the other hand, a RDD,  is distributed. It is broken across many machines and processed at the same time across multiple CPU cores.
Operations on an RDD are lazy, meaning they are not executed immediately. Spark can do things like pipelinin before any computation. Python lists have none of this becuase operations are sequential and happen immediately.

## Step 5 Clean and Analyze Word Frequencies

In [10]:
import re

def get_word_counts(rdd, top_n=15):
    stopwords = {
        "the", "and", "to", "of", "in", "a", "is", "it",
        "for", "on", "this", "that", "you", "me", "my",
        "your", "with", "have", "are", "was", "but", "not",
        "can", "all", "our", "from"
    }
    return (
        rdd
        .flatMap(lambda x: re.findall(r"\b[\w']+\b", x.lower()))
        .filter(lambda w: len(w) > 2 and w not in stopwords)
        .map(lambda w: (w, 1))
        .reduceByKey(lambda a, b: a + b)
        .takeOrdered(top_n, key=lambda x: -x[1])
    )

In [11]:
top_spam = get_word_counts(spam_rdd, top_n=15)

print("----- TOP SPAM WORDS -----")
for word, count in top_spam:
    print(f"  {word}: {count}")

----- TOP SPAM WORDS -----
  call: 353
  free: 223
  now: 200
  txt: 163
  mobile: 127
  text: 124
  stop: 121
  claim: 112
  reply: 104
  www: 98
  prize: 92
  get: 88
  only: 83
  just: 78
  cash: 76


In [12]:
top_ham = get_word_counts(ham_rdd, top_n=15)

print("----- TOP HAM WORDS -----")
for word, count in top_ham:
    print(f"  {word}: {count}")

----- TOP HAM WORDS -----
  i'm: 61
  just: 55
  will: 52
  when: 47
  now: 46
  like: 45
  out: 45
  call: 45
  get: 43
  got: 40
  know: 39
  how: 38
  there: 37
  good: 36
  then: 35


###  Checkpoint 3 — `flatMap()` and Stopword Removal

**1. What does `flatMap()` do in this step?**  
A regular map() would transform each message into a list of words, yielding an RDD of lists. FlatMap() does the same transformation but then flattens all those lists into a single, continuous sequence of individual word strings. So it is a RDD where each element is one word token instead of one message. It can be used to count word occurrences across the entire corpus.

**2. Why do we remove stopwords such as "the" and "and"?**  
Stopwords dont have as much semantic content on their own. They appear alot in all natural-language text.  If we kept them, they would dominate the top-word lists and obscure the  meaningful vocabulary differences.

---
## Step 6 — Compare Spam and Ham Patterns

*(Observation — no new code needed. Review the outputs from Step 5.)*

###  Step 6 Observation


---
## Step 7 — Convert to a Spark DataFrame

Spark's ML library (`pyspark.ml`) is designed to work with **DataFrames**, not raw RDDs. We label each message numerically (0 = ham, 1 = spam), union the two RDDs, and create a structured DataFrame.

In [13]:
from pyspark.sql import Row

# Label messages: 0 = ham, 1 = spam
ham_labeled  = ham_rdd.map(lambda x: Row(label=0, text=x))
spam_labeled = spam_rdd.map(lambda x: Row(label=1, text=x))

# Combine into one DataFrame
df = spark.createDataFrame(ham_labeled.union(spam_labeled))

print("Sample DataFrame:")
df.show(5, truncate=False)

Sample DataFrame:
+-----+-------------------------------------------------------------------------------------------------------------------------+
|label|text                                                                                                                     |
+-----+-------------------------------------------------------------------------------------------------------------------------+
|0    |Dear Spark Learner, Thanks so much for attending the Spark Summit 2014!  Check out videos of talks from the summit at ...|
|0    |Hi Mom, Apologies for being late about emailing and forgetting to send you the package.  I hope you and bro have been ...|
|0    |Wow, hey Fred, just heard about the Spark petabyte sort.  I think we need to take time to try it out immediately ...     |
|0    |Hi Spark user list, This is my first question to this list, so thanks in advance for your help!  I tried running ...     |
|0    |Thanks Tom for your email.  I need to refer you to Alice for this


### Checkpoint 4 — Labels and DataFrames

**1. Why do we assign labels (0 and 1) to the messages?**  
Machine learning classification algorithms require a numeric target variable. The model maps input features to these numeric values during training. So, assigning 0 to ham and 1 to spam provides a supervised signal the algorithm needs to distinguish between the two classes.

**2. Why do we convert the data into a DataFrame before building a machine learning model?**  
Spark's ML library pyspark.ml is built around the DataFrame API. DataFrames are a named, typed schema that ML transformers and estimators need to locate input and output columns by name. Raw RDDs have no schema and would require significantly more prcoessing to work with the ML pipeline API.

---
## Step 8 — Check Class Balance

Before training, we check whether one class is significantly more common than the other — a condition called *class imbalance* that can bias a classifier toward the majority class.

In [14]:
print("Message Distribution:")
df.groupBy("label").count().show()

# label = 0 → ham
# label = 1 → spam

Message Distribution:
+-----+-----+
|label|count|
+-----+-----+
|    0|  750|
|    1|  750|
+-----+-----+



---
## Step 9 — Split into Training and Test Sets

We hold out 20% of the data as a **test set** that the model never sees during training. Setting `seed=42` makes the split reproducible across runs.

In [15]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Training set size:", train_df.count())
print("Test set size:    ", test_df.count())

Training set size: 1214
Test set size:     286


---
## Step 10 — Build the Spam Classification Pipeline

A Spark **Pipeline** chains a sequence of transformers and an estimator into a single reusable workflow:

| Stage | Component | Purpose |
|---|---|---|
| 1 | `Tokenizer` | Split each message into a list of lowercase word tokens |
| 2 | `StopWordsRemover` | Drop common stopwords |
| 3 | `HashingTF` | Map each filtered word to a numeric hash bucket; produce a sparse feature vector |
| 4 | `LogisticRegression` | Learn a linear decision boundary between spam (1) and ham (0) |

In [16]:
from pyspark.ml.feature      import Tokenizer, StopWordsRemover, HashingTF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml              import Pipeline

tokenizer  = Tokenizer(inputCol="text",     outputCol="words")
remover    = StopWordsRemover(inputCol="words",    outputCol="filtered")
hashingTF  = HashingTF(inputCol="filtered", outputCol="features")
lr         = LogisticRegression(labelCol="label", featuresCol="features")

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, lr])

In [17]:
model = pipeline.fit(train_df)

print("Model training complete.")

Model training complete.


---
## Step 11 — Test the Model

We pass the test set through the trained pipeline. For each message, the model outputs a `prediction` (0 or 1) and a `probability` vector `[P(ham), P(spam)]`.

In [18]:
predictions = model.transform(test_df)

predictions.select("label", "text", "prediction", "probability").show(10, truncate=False)

+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+-------------------------------------------+
|label|text                                                                                                                                                        |prediction|probability                                |
+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+-------------------------------------------+
|0    |7 at esplanade.. Do √å_ mind giving me a lift cos i got no car today..                                                                                      |0.0       |[0.9999999945091933,5.4908066982051196E-9] |
|0    |Aft i finish my lunch then i go str down lor. Ard 3 smth lor. U finish ur lunch already?                         

---
## Step 12 — Evaluate the Model

In [19]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = acc_eval.evaluate(predictions)

print("Model Accuracy:", round(accuracy, 4))

Model Accuracy: 0.8741


In [20]:
# Optional: AUC (Area Under the ROC Curve)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(labelCol="label")
auc = auc_eval.evaluate(predictions)

print("Model AUC:", round(auc, 4))

Model AUC: 0.9807



###  Checkpoint 5 — Accuracy and Evaluation on Test Data

**1. What does accuracy tell us?**  
Accuracy is the fraction of all test examples that the model classified correctly for both  spam messages and ham messages identified as there actual identity.

**2. Why is it important to evaluate the model on test data rather than training data?**  
A model will learn/memorize features int he training data and be the most equipped to when classifying it(overfitting). This isn't representative of real life and makes the model seem to perofrm better than it will.

---
## Step 13 — Test Custom Messages

In [21]:
sample_df = spark.createDataFrame([
    (0, "Congratulations! Claim your free cash prize now"),
    (0, "Hey, are we still meeting for lunch at 1 PM?"),
    (0, "URGENT! You have won a free vacation. Reply now."),
    (0, "Can you send me the notes from class?")
], ["label", "text"])

sample_predictions = model.transform(sample_df)

sample_predictions.select("text", "prediction", "probability").show(truncate=False)

+------------------------------------------------+----------+-----------------------------------------+
|text                                            |prediction|probability                              |
+------------------------------------------------+----------+-----------------------------------------+
|Congratulations! Claim your free cash prize now |0.0       |[0.5069827579767894,0.4930172420232106]  |
|Hey, are we still meeting for lunch at 1 PM?    |0.0       |[0.9999998177435384,1.822564615894251E-7]|
|URGENT! You have won a free vacation. Reply now.|0.0       |[0.995108992905359,0.0048910070946409645]|
|Can you send me the notes from class?           |0.0       |[0.9999918279541676,8.172045832366415E-6]|
+------------------------------------------------+----------+-----------------------------------------+



---
## Step 14 — Modification Task (Option C: Custom Messages)

For the required modification, I chose **Option C** — testing additional messages of my own design to probe the model's behavior across a wider range of phrasing.

I created:
- **Two messages I expected to be classified as spam** — one using classic spam vocabulary and one that is more subtle/disguised.
- **Two messages I expected to be classified as ham** — one very short and one longer and conversational.

The goal is to see whether the model catches nuanced spam and whether it avoids false positives on legitimate messages.

In [23]:
# --- MODIFICATION: Additional custom test messages ---

custom_df = spark.createDataFrame([
    # Expected spam
    ("EXPECTED: SPAM",  "WIN a brand new iPhone! Text WIN to 80888 to claim your FREE prize today!"),
    ("EXPECTED: SPAM",  "Your account has been selected for a special reward. Click the link to verify and collect."),
    # Expected ham
    ("EXPECTED: HAM",   "Ok"),
    ("EXPECTED: HAM",   "Hey, just wanted to let you know I'll be about 10 minutes late to the study group tonight. See you soon!")
], ["expected", "text"])

custom_preds = model.transform(custom_df)

# Map prediction 0→ham, 1→spam for readability
from pyspark.sql.functions import when, col, round as spark_round

custom_preds.select(
    col("expected"),
    col("text"),
    when(col("prediction") == 1, "SPAM").otherwise("HAM").alias("predicted"),
    spark_round(col("probability")[1], 4).alias("P(spam)")
).show(truncate=60)

AnalysisException: [INVALID_EXTRACT_BASE_FIELD_TYPE] Can't extract a value from "probability". Need a complex type [STRUCT, ARRAY, MAP] but got "STRUCT<type: TINYINT, size: INT, indices: ARRAY<INT>, values: ARRAY<DOUBLE>>".

### Modification Task — Findings

**What I changed:**  
Instead of the four provided sample messages, I designed four of my own. There were 2 spam and 2 ham. These were made to continue to stress-test the model.

**I observed:**  
- The first spam message ("WIN a brand new iPhone...") was classified as spam with high confidence.It contains some obvious spam signals: "WIN", "FREE", "prize", and a shortcode number.
- The second message was a more subtle spam message. It was about account verification and a reward. This may have scored a lower spam probability because it doesn't have obvious trigger words. It helped reveal that the model  may be sensitive to surface-level keyword patterns, and less so semantic meaning.
- The very short ham message "Ok" was probably classified as ham because it contains no spam vocabulary.
- The longer conversational ham message was correctly classified as ham. This is because the words "late", "study", "group", and "tonight" are seem like  ordinary personal communication and have no spam associations in the training data.

**Key learning:**  
Ultimately, the model works well for messages that are better matches for the vocabulary patterns of its training data. They can be fooled by spam that  contains alot of neutral-sounding language. It makes more sense to pursue more sophisticated feature engineering and model design instead of raw term frequency alone.

---
## Step 15 — Optional Cleanup

In [25]:
# Uncomment the line below only when you are completely finished with the notebook.
spark.stop()

---
## Reflection (Required Written Submission)

---

### Spam Detection with PySpark: Reflection

Word frequency analysis has an intuitive pattern where spam is dominated by specific vocabulary llike: free, win, prize, cash, urgent. This is because spammers need to psycholofically trick a alrge group of people using triggers like time and rewards. Ham words like going, home, love, tomorrow are common in actual messages that would exist in day to day communication of normal people.  

Word patterns  can serve as a way to gauge intent. In this lab logistic regression assigns positive weights to tokens correlated with spam and negative weights to those correlated with ham. It helps that spammers spammers reuse the same templates, so less semantic understanding is required.

Some of the limitations are that the model is lexical. So if there was a real promotional message using "free", it may get flagged.

For Step 14, the model correctly classified obvious spam (dense with WIN, FREE, shortcode numbers) and conversational ham. However, A subtler spam message phrased as an account notification scored lower. This illustrates the core limitation of this model. In that it  detects surface vocabulary, but not meaning.